# ARK-Innovation — a quantitative teardown 🔬
### Real ARKK tape · trend/momentum/reversion overlay race · dollar-weighted IRR · HAC inference · block-bootstrap CIs · capacity

![Signal: Mixed](https://img.shields.io/badge/Signal-Mixed-dab617?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![A_buy--the--top_machine%3F: Confirmed](https://img.shields.io/badge/A_buy--the--top_machine%3F-Confirmed-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim now carrying its standard error.* We race three long/flat overlays on ARKK's real daily tape under autocorrelation-robust inference, then turn to the headline: the time-weighted vs dollar-weighted return gap (Dichev 2007) that decides what ARKK's investors actually earned.

> ⚠️ **Not investment advice.** Data: Yahoo! daily auto-adjusted (total-return-proxy) closes for ARKK / QQQ / XLK since 2014-10-31, cache-first. Real headline numbers are pinned in [docs/results.md](../docs/results.md) (as-of 2026-06-18); methods in [`docs/references.md`](../docs/references.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back into intuition. House style in [METHODOLOGY.md](../../../METHODOLOGY.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # the study package
sys.path.insert(0, os.path.abspath("../../.."))    # repo root (quantlab/)
%matplotlib inline
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9.5, 5.0), "axes.grid": True,
                     "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
RED, AMBER, GREEN, GREY = "#c0392b", "#dab617", "#2ea44f", "#8b949e"

from ark_innovation import data, strategy as st

TICKERS = ["ARKK", "QQQ", "XLK"]
CUT = pd.Timestamp("2026-06-01")  # exclusive — drop the partial month

def _have_cache():
    return all(os.path.exists(data._cache_path(t, data.DEFAULT_CACHE)) for t in TICKERS)

HAVE_REAL = _have_cache()

def real_close(t):
    px = data.fetch_prices(t, fetch=False)
    return px["close"][px.index < CUT]

print("real daily cache present:", HAVE_REAL)

# Offline fallback: a deterministic boom-bust tape stands in for the real ARKK series.
# (Clearly bannered as synthetic; real headline numbers are quoted from docs/results.md.)
def synthetic_arkk():
    fr, _ = data.synthetic_hype_cycle(n_months=180, bust=0.04, chase=15.0, seed=334)
    # upsample the monthly NAV to a daily-ish business index for the overlay charts
    daily = data.synthetic_daily(n_days=2910, momentum=0.05, drift=0.0003, seed=334)
    return daily["close"]


real daily cache present: True


## Verdict, up front

| Axis | Stamp | The decisive number |
|---|---|---|
| **Signal** | Mixed | Trend 50/200 HAC *t* = **+2.09** (CI low **+0.22 bps**); ts-momentum *t* = +1.36, mean-rev *t* = +0.33 → None |
| **Tradability** | Mirage | Overlay excess Sharpe over ARKK B&H **+0.07**; vs QQQ B&H Sharpe **0.93** it loses outright |
| **Buy-the-top machine?** | Confirmed | +737% boom, -81% bust; dollar-weighted return deeply negative (~\$7–10bn destroyed) |

> 💡 **In plain words.** One trend rule technically clears the bar, but only by going to cash in the crash; nothing here beats an index fund; and the average dollar in ARKK lost money even though the price went up.

## 1 · The claim, steelmanned

Three testable hypotheses:

- **H₁ (momentum).** ARKK's daily returns carry tradable persistence — a trend / time-series-momentum overlay earns a positive risk-adjusted return with HAC *t* ≥ 2.
- **H₂ (tradable edge).** That overlay beats *both* ARKK buy-and-hold and a QQQ buy-and-hold on an excess-of-cash Sharpe basis.
- **H₃ (investor experience).** The dollar-weighted (money-weighted IRR) return ARKK investors earned is far below its time-weighted (buy-and-hold) return — the performance-chasing penalty (Dichev 2007; Friesen-Sapp 2007).

## 2 · So what? — what rides on each answer

H₁/H₂ test whether *innovation momentum* was a real, harvestable edge or just beta you could have bought cheaper. H₃ tests whether the **fund's posted return** (the number ARK advertised) bears any resemblance to **investors' realised return** — the gap is the literature's cleanest indictment of thematic-ETF hype (Ben-David et al. 2023).

## 3 · How we'd know — the protocol

1. **Decompose** ARKK's arc into boom (→2021 peak) and bust segments — an exact identity, no fitting.
2. **Robust inference** — Newey-West HAC *t* on each overlay's daily return; circular block-bootstrap CI (preserves volatility clustering).
3. **Alpha vs beta** — excess-of-cash Sharpe race vs ARKK and QQQ buy-and-hold (a long/flat overlay is part-time in cash, so we compare excess-to-excess).
4. **Capacity / cost** — one-way × NAV cost sweep, one documented execution lag (signal at close *t* → return *t+1*).
5. **The behaviour gap** — money-weighted IRR vs time-weighted CAGR, with the mechanism validated on a deterministic synthetic positive control.

## 4 · The teardown

### 4a · The boom-bust decomposition

In [2]:
if HAVE_REAL:
    c = real_close('ARKK'); tape = 'REAL ARKK (cache)'
else:
    c = synthetic_arkk(); tape = 'SYNTHETIC fallback (offline)'
pk = c.idxmax()
boom = st.time_weighted_return(c[c.index <= pk], st.TRADING_DAYS_PER_YEAR)
bust = st.time_weighted_return(c[c.index >= pk], st.TRADING_DAYS_PER_YEAR)
full = st.time_weighted_return(c, st.TRADING_DAYS_PER_YEAR)
print('tape:', tape)
print(f'  inception -> peak ({pk.date()}): {boom["cagr"]*100:+.1f}%/yr  ({boom["total_return"]*100:+.0f}%)')
print(f'  peak -> end:                 {bust["cagr"]*100:+.1f}%/yr  ({bust["total_return"]*100:+.0f}%)')
print(f'  full sample:                 {full["cagr"]*100:+.1f}%/yr  ({full["total_return"]*100:+.0f}%)')
print('  pinned real: boom +40.3%/yr (+737%), bust -11.3%/yr (-47%), full +13.8%/yr')

tape: REAL ARKK (cache)
  inception -> peak (2021-02-12): +40.3%/yr  (+737%)
  peak -> end:                 -11.3%/yr  (-47%)
  full sample:                 +13.8%/yr  (+346%)
  pinned real: boom +40.3%/yr (+737%), bust -11.3%/yr (-47%), full +13.8%/yr


### 4b · The overlay race under HAC inference + block-bootstrap CI

> 💡 **In plain words.** Does any simple timing rule on ARKK beat the noise? We charge each one a robust *t*-stat and a bootstrap confidence interval.

In [3]:
rows = []
overlays = {'trend 50/200': st.ma_crossover_signal(c, 50, 200),
            'ts-momentum 1d': st.ts_momentum_signal(c, 1),
            'mean-rev z<-1': st.mean_reversion_signal(c, 5, 1.0)}
for name, sig in overlays.items():
    d = st.overlay_returns(c, sig, cost_bps=0.0)
    s = st.summarize_overlay(d, 'gross')
    lo, hi = st.block_bootstrap_ci(d['gross'].to_numpy(), block=21, n_boot=1000)
    rows.append((name, round(s['mean_bps'], 2), round(s['sharpe'], 2),
                 round(s['hac_t'], 2), f'[{lo*1e4:+.2f},{hi*1e4:+.2f}]', f"{s['pct_invested']*100:.0f}%"))
bh = c.pct_change().fillna(0.0).to_numpy()
rows.append(('ARKK buy&hold', round(np.nanmean(bh)*1e4, 2), round(st.sharpe(bh), 2),
             round(st.hac_tstat(bh), 2), '-', '100%'))
tbl = pd.DataFrame(rows, columns=['overlay', 'mean bps/d', 'Sharpe', 'HAC t', '95% CI bps', '% inv'])
print('tape:', tape); display(tbl)
print('(pinned real: trend t=+2.09 CI [+0.22,+12.30], ts-mom t=+1.36, mr t=+0.33)')

tape: REAL ARKK (cache)


,overlay,mean bps/d,Sharpe,HAC t,95% CI bps,% inv
0,trend 50/200,6.45,0.60,2.09,"[+0.49,+12.38]",58%
1,ts-momentum 1d,4.16,0.39,1.36,"[-1.57,+10.10]",53%
2,mean-rev z<-1,0.74,0.09,0.33,"[-3.22,+4.85]",24%
3,ARKK buy&hold,8.04,0.53,1.88,-,100%


(pinned real: trend t=+2.09 CI [+0.22,+12.30], ts-mom t=+1.36, mr t=+0.33)


### 4c · Alpha vs beta — the excess-of-cash Sharpe race

> 💡 **In plain words.** The trend overlay sits in cash ~42% of the time, so comparing its raw Sharpe to a fully-invested index is unfair. We subtract cash from both and race it against ARKK *and* QQQ buy-and-hold.

In [4]:
trend = st.overlay_returns(c, st.ma_crossover_signal(c, 50, 200), 0.0)['gross'].to_numpy()
arkk_bh = c.pct_change().fillna(0.0).to_numpy()
if HAVE_REAL:
    q = real_close('QQQ'); qqq_bh = q.pct_change().fillna(0.0).to_numpy()
    m = min(len(trend), len(qqq_bh)); src = 'REAL QQQ'
else:
    qqq_bh = data.synthetic_daily(n_days=len(c), momentum=0.0, drift=0.0006, seed=7)['close'].pct_change().fillna(0.0).to_numpy()
    m = min(len(trend), len(qqq_bh)); src = 'SYNTHETIC QQQ stand-in'
vs_arkk = st.excess_sharpe(trend, arkk_bh)
vs_qqq = st.excess_sharpe(trend[:m], qqq_bh[:m])
print('benchmark source:', src)
print(f'  trend overlay excess Sharpe : {vs_arkk["strat_excess_sharpe"]:.2f}')
print(f'  ARKK buy&hold excess Sharpe : {vs_arkk["bench_excess_sharpe"]:.2f}  (diff {vs_arkk["diff"]:+.2f})')
print(f'  QQQ  buy&hold excess Sharpe : {vs_qqq["bench_excess_sharpe"]:.2f}  (diff {vs_qqq["diff"]:+.2f})')
print('  pinned real: overlay vs ARKK diff +0.07; QQQ B&H Sharpe 0.93')

benchmark source: REAL QQQ
  trend overlay excess Sharpe : 0.60
  ARKK buy&hold excess Sharpe : 0.53  (diff +0.07)
  QQQ  buy&hold excess Sharpe : 0.93  (diff -0.33)
  pinned real: overlay vs ARKK diff +0.07; QQQ B&H Sharpe 0.93


### 4d · The behaviour gap — time-weighted vs dollar-weighted, with a positive control

> 💡 **In plain words.** The IRR machinery: what did the *average dollar* earn, given when the money arrived? We plant a known gap on a synthetic fund and recover it (a machinery proof — never market evidence), then state the real figure from the literature.

In [5]:
rows = []
for label, bust_, chase in [('null: steady grower', 0.0, 0.0),
                            ('boom-bust + chasing', 0.04, 15.0)]:
    fr, _ = data.synthetic_hype_cycle(n_months=180, bust=bust_, chase=chase, seed=334)
    g = st.behaviour_gap(fr['price'], fr['flow'], st.MONTHS_PER_YEAR)
    rows.append((label, round(g['twr_cagr']*100, 1), round(g['dwr_irr']*100, 1), round(g['gap']*100, 1)))
tbl = pd.DataFrame(rows, columns=['synthetic fund', 'time-weighted %/yr', 'dollar-weighted %/yr', 'gap pts'])
display(tbl)
print('SYNTHETIC positive control — the IRR engine recovers the planted gap.')
print('Real ARKK (literature): fund time-weighted +13.8%/yr, but the dollar-weighted')
print('return was deeply negative — Morningstar est. ~$7-10bn of investor wealth destroyed.')

,synthetic fund,time-weighted %/yr,dollar-weighted %/yr,gap pts
0,null: steady grower,11.6,15.7,-4.1
1,boom-bust + chasing,1.9,-4.8,6.6


SYNTHETIC positive control — the IRR engine recovers the planted gap.
Real ARKK (literature): fund time-weighted +13.8%/yr, but the dollar-weighted
return was deeply negative — Morningstar est. ~$7-10bn of investor wealth destroyed.


### 4e · The momentum engine's own positive control

Sanity check that the overlay machinery *can* detect momentum when it exists — so its near-silence on real ARKK is a finding, not a broken pipeline.

In [6]:
for label, mom in [('null: no momentum', 0.0), ('planted momentum', 0.20)]:
    s = data.synthetic_daily(n_days=3000, momentum=mom, drift=0.0, seed=334)
    d = st.overlay_returns(s['close'], st.ts_momentum_signal(s['close'], 1), 0.0)
    t = st.summarize_overlay(d, 'gross')['hac_t']
    print(f'  {label:20s} HAC t = {t:+.2f}')
print('  (pinned: null t=-0.15, planted t=+5.24)')

  null: no momentum    HAC t = -0.15


  planted momentum     HAC t = +5.24
  (pinned: null t=-0.15, planted t=+5.24)


## 5 · The verdict

- **Signal — Mixed.** Trend 50/200 clears the bar (HAC *t* = +2.09, block-bootstrap CI [+0.22, +12.30] bps) but only as crash avoidance; ts-momentum (+1.36) and mean-reversion (+0.33) are None. *Real on the slow trend · None on momentum/reversion.*
- **Tradability — Mirage.** Overlay excess Sharpe over ARKK B&H is +0.07, and it loses outright to QQQ B&H (Sharpe 0.93). No investable edge.
- **Buy-the-top machine? — Confirmed.** Fund +13.8%/yr, but a +737% boom into a -81% bust with assets concentrated near the top → dollar-weighted return deeply negative.

## 6 · Could you trade it?

The trend overlay is robust to costs (still HAC *t* = +2.05 at 20 bps round-trip) because it trades rarely — but cost was never the binding constraint. **Capacity and selection are:** the only value the rule adds is sidestepping a drawdown a diversified QQQ holder (Sharpe 0.93, +19.7%/yr) never suffered. You'd be taking concentrated single-fad risk to *underperform* an index fund. The honest break-even isn't a cost level — it's the realisation that the trade shouldn't be put on at all.

## 7 · Going further

- **Real flows.** Drop ARKK's monthly shares-outstanding into `data.fetch_flows` to replace the literature estimate with a computed dollar-weighted IRR off this exact engine.
- **Cross-section.** Run the behaviour gap across the thematic-ETF universe (opt into the survivorship guard — dead thematic ETFs bias the gap *downward*, so the survivors understate it) to test Ben-David et al.'s product-level buy-the-top claim.
- **Regime split.** The trend overlay's whole edge is one crash; test it pre-2021 alone (it should vanish) to confirm it's crash-insurance, not timing skill.